# Tarea 7 · Tipos de datos y columnas calculadas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/feat/7-tipos-columnas/notebooks/trabajo/7_tipos_columnas.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Matías.

Tomamos `data/processed/telco_step1.csv`, generado en la tarea 6, adecuamos los tipos, eliminamos el identificador del cliente y creamos dos columnas por aritmética directa. La salida es `data/processed/telco_step2.csv`, entrada de la tarea 9. La evaluación de outliers corresponde a la tarea 8.

## Cómo ejecutarlo

- **Local:** instalar `requirements.txt`, abrir este notebook desde el repositorio y reiniciar el entorno y ejecutar todas las celdas en orden. La carga busca `telco_step1.csv` desde el directorio de ejecución y sus padres.
- **Colab:** abrir el badge y elegir **Entorno de ejecución → Reiniciar sesión y ejecutar todo**, o las opciones equivalentes de reiniciar y ejecutar todas las celdas. La entrada se lee desde `main` de GitHub, donde la tarea 6 ya está integrada. No hace falta montar Drive ni subir el CSV.
- La salida queda en `data/processed/` dentro del directorio de trabajo. En Colab se puede descargar desde el panel **Archivos**.
- El badge usa la rama `feat/7-tipos-columnas` durante la revisión: estará disponible después del push. Al integrar la tarea, actualizar el enlace para que apunte a `main`.


## 1. Carga de la salida de la tarea 6

Usamos el archivo intermedio acordado, sin repetir la limpieza de `TotalCharges` ni modificar los datos originales. Guardamos una copia para verificar al final que se mantengan las filas y las variables que no deben cambiar. El resumen de tipos permite observar qué campos Pandas interpreta como texto o números al cargar el CSV.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

GITHUB_REPO = 'julianschmidt1/tpi-analisis-de-datos'
GITHUB_REF = 'main'
RUTA_ENTRADA = Path('data/processed/telco_step1.csv')
URL_ENTRADA = (
    f'https://raw.githubusercontent.com/{GITHUB_REPO}/'
    f'{GITHUB_REF}/{RUTA_ENTRADA.as_posix()}'
)

if EN_COLAB:
    RAIZ = Path.cwd()
    fuente = URL_ENTRADA
else:
    actual = Path.cwd().resolve()
    RAIZ = next(
        (ruta for ruta in (actual, *actual.parents) if (ruta / RUTA_ENTRADA).is_file()),
        None,
    )
    if RAIZ is None:
        raise FileNotFoundError(
            f'No se encontró {RUTA_ENTRADA.as_posix()}. '
            'Ejecutá la tarea 6 o actualizá el repositorio y abrí Jupyter dentro de él.'
        )
    fuente = RAIZ / RUTA_ENTRADA

df_step1 = pd.read_csv(fuente)
df = df_step1.copy()
assert df['TotalCharges'].dtype == 'float64'
assert df['TotalCharges'].notna().all(), 'La entrada debe cumplir la tarea 6.'

print(f'Entorno: {"Colab" if EN_COLAB else "local"}')
print(f'Fuente: {fuente}')
print(f'Entrada: {df.shape[0]:,} clientes y {df.shape[1]} columnas.')
df.info()


Entorno: local
Fuente: D:\src\tpi-analisis-de-datos\data\processed\telco_step1.csv
Entrada: 7,043 clientes y 21 columnas.
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 no

La entrada tiene 7.043 clientes y 21 columnas. `TotalCharges` ya es numérico y sus faltantes fueron tratados en la tarea 6. `SeniorCitizen` representa una condición binaria aunque se almacene como entero; las variables textuales representan categorías. Adecuar sus tipos facilita agrupar clientes sin interpretar códigos como magnitudes.


## 2. Adecuación de tipos y eliminación del identificador

Eliminamos `customerID` porque es un identificador y no una característica del servicio o comportamiento. Convertimos `SeniorCitizen` de `0/1` a `No/Yes` para expresar su naturaleza categórica y mantener una representación coherente con otros campos binarios. Verificamos el dominio antes de mapearlo para evitar generar nulos silenciosamente.

Las 17 variables categóricas, incluido `Churn`, quedan como `category`. Se conservan categorías como `No internet service` y `No phone service`, porque describen situaciones distintas de no contratar una prestación sobre un servicio disponible. `Contract` queda como categoría sin codificar: su orden y codificación se resuelven en la tarea 22.

Mantenemos los nombres originales de las columnas restantes para respetar el contrato de integración, especialmente `tenure`, `MonthlyCharges`, `TotalCharges` y `Churn`. La antigüedad se expresa como entero y los importes como `float64`.


In [2]:
assert df['SeniorCitizen'].isin([0, 1]).all(), 'SeniorCitizen tiene valores inesperados.'
df = df.drop(columns='customerID')
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

columnas_categoricas = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
    'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn',
]
df[columnas_categoricas] = df[columnas_categoricas].astype('category')
df = df.astype({
    'tenure': 'int64',
    'MonthlyCharges': 'float64',
    'TotalCharges': 'float64',
})

print(f'Variables categóricas: {len(columnas_categoricas)}')
print(f'Valores de SeniorCitizen: {df["SeniorCitizen"].cat.categories.tolist()}')
print(f'Valores de Churn: {df["Churn"].cat.categories.tolist()}')


Variables categóricas: 17
Valores de SeniorCitizen: ['No', 'Yes']
Valores de Churn: ['No', 'Yes']


## 3. Columnas calculadas por aritmética directa

Creamos las dos variables propuestas en el plan:

- **`ingreso_estimado = MonthlyCharges * tenure`:** aproxima el total de cargos suponiendo que el cargo mensual actual se mantuvo constante durante toda la antigüedad del cliente. Es una aproximación, no el ingreso efectivamente observado.
- **`diferencia_facturacion = TotalCharges - ingreso_estimado`:** compara el total registrado con esa aproximación. Un valor positivo indica un total registrado mayor; uno negativo indica un total menor. Puede reflejar cambios de plan, promociones u otras condiciones que el dataset no permite distinguir; por sí solo no demuestra un error de facturación.

Mostramos cinco clientes para verificar visualmente las relaciones entre las columnas de origen y las calculadas.


In [3]:
df['ingreso_estimado'] = df['MonthlyCharges'] * df['tenure']
df['diferencia_facturacion'] = df['TotalCharges'] - df['ingreso_estimado']

columnas_facturacion = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'ingreso_estimado', 'diferencia_facturacion',
]
display(df[columnas_facturacion].head())


,tenure,MonthlyCharges,TotalCharges,ingreso_estimado,diferencia_facturacion
0,1,29.85,29.85,29.85,0.00
1,34,56.95,1889.50,1936.30,-46.80
2,2,53.85,108.15,107.70,0.45
3,45,42.30,1840.75,1903.50,-62.75
4,2,70.70,151.65,141.40,10.25


La tabla muestra que el total registrado puede diferir de mantener el cargo mensual actual durante todos los meses. Por eso, las nuevas columnas sirven para explorar la relación entre antigüedad y facturación y requieren interpretar la suposición de cargo constante.

Para los clientes de antigüedad cero cuyo `TotalCharges` se imputó a cero en la tarea 6, ambas columnas calculadas también quedan en cero. Las diferencias negativas se conservan: no son valores inválidos por definición.

Estas columnas se derivan de variables existentes y pueden ser redundantes. Para predecir `TotalCharges` en la regresión lineal no se debe usar `diferencia_facturacion`, porque contiene el propio objetivo; el plan utiliza solo `tenure` como predictor.


## 4. Verificación del esquema de salida

Definimos los tipos esperados para las 22 columnas: 17 categóricas y 5 numéricas. Comprobamos que no se hayan eliminado clientes, que no queden nulos, que `Churn` conserve las etiquetas y que las variables originales restantes mantengan sus valores. Finalmente ejecutamos `df.info()` para comprobar el criterio de terminado de la tarea.


In [4]:
tipos_esperados = {
    **dict.fromkeys(columnas_categoricas, 'category'),
    'tenure': 'int64',
    'MonthlyCharges': 'float64',
    'TotalCharges': 'float64',
    'ingreso_estimado': 'float64',
    'diferencia_facturacion': 'float64',
}
assert len(df) == len(df_step1), 'No deben eliminarse clientes en esta tarea.'
assert set(df.columns) == set(tipos_esperados), 'El esquema de columnas cambió.'
assert not df.isna().any().any(), 'La transformación no debe generar nulos.'
assert df['Churn'].isin(['Yes', 'No']).all()
assert all(str(df[columna].dtype) == tipo for columna, tipo in tipos_esperados.items())

columnas_sin_cambios = df_step1.columns.drop(['customerID', 'SeniorCitizen'])
pd.testing.assert_frame_equal(
    df[columnas_sin_cambios].astype(object),
    df_step1[columnas_sin_cambios].astype(object),
)
assert df['SeniorCitizen'].astype(object).equals(
    df_step1['SeniorCitizen'].map({0: 'No', 1: 'Yes'}).astype(object)
)

df.info()
print(f'Esquema validado: {len(df):,} clientes y {df.shape[1]} columnas.')


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   gender                  7043 non-null   category
 1   SeniorCitizen           7043 non-null   category
 2   Partner                 7043 non-null   category
 3   Dependents              7043 non-null   category
 4   tenure                  7043 non-null   int64   
 5   PhoneService            7043 non-null   category
 6   MultipleLines           7043 non-null   category
 7   InternetService         7043 non-null   category
 8   OnlineSecurity          7043 non-null   category
 9   OnlineBackup            7043 non-null   category
 10  DeviceProtection        7043 non-null   category
 11  TechSupport             7043 non-null   category
 12  StreamingTV             7043 non-null   category
 13  StreamingMovies         7043 non-null   category
 14  Contract                7043 non-nu

## 5. Exportación y recuperación de los tipos

Guardamos `data/processed/telco_step2.csv` sin índice, conservando las filas y el orden de las columnas. Esta salida permite a la tarea 9 continuar con la revisión de calidad y la decisión sobre outliers de la tarea 8.

**CSV no conserva los tipos `category`.** Para recuperarlos al continuar el análisis, se puede copiar `columnas_categoricas` y `tipos_esperados` de este notebook y cargar el archivo con `pd.read_csv(ruta, dtype=tipos_esperados)`. La siguiente celda realiza esa lectura y compara el resultado con `df` para verificar que el archivo exportado pueda reconstruirse. Usamos tolerancia numérica para las diferencias de precisión decimal propias del CSV.


In [5]:
RUTA_SALIDA = RAIZ / 'data/processed/telco_step2.csv'
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RUTA_SALIDA, index=False, encoding='utf-8')

df_recargado = pd.read_csv(RUTA_SALIDA, dtype=tipos_esperados)
pd.testing.assert_frame_equal(df, df_recargado, check_exact=False, rtol=1e-12, atol=1e-9)
print(f'Archivo generado: {RUTA_SALIDA.relative_to(RAIZ).as_posix()}')
print(f'Salida: {len(df_recargado):,} clientes y {df_recargado.shape[1]} columnas.')
print('Verificación de lectura: valores y tipos recuperados correctamente.')


Archivo generado: data/processed/telco_step2.csv
Salida: 7,043 clientes y 22 columnas.
Verificación de lectura: valores y tipos recuperados correctamente.


**Resultado de la tarea:** se conservan 7.043 clientes, se elimina `customerID`, `SeniorCitizen` queda como `No/Yes` categórico y se incorporan las dos variables de facturación. El DataFrame queda con 22 columnas y sin nulos. Los tipos categóricos se recuperan al cargar el CSV indicando el esquema. El dataset limpio final se exportará en la tarea 9.
